# gridgen vs Dan's conformal solver, on the same outlines in the same plane

Companion to `04b_gridgen_vs_dan.md`. Runs **both** solvers in one Python process:

- **gridgen** (Sakov's C library, Schwarz–Christoffel via CRDT) through `pygridgen`;
- **Dan's finite-element solver**, `mom6_forge/_conformal.py` from the PR #151 worktree, loaded
  **by file path** (it needs only numpy, scipy, shapely, matplotlib and pyproj), so the
  `mom6_forge` package itself is not imported.

**Kernel.** Run this in the `dev_gridgen` env (Python 3.12, pygridgen 0.3.0 from conda-forge,
`setuptools<81` pinned because pygridgen imports `pkg_resources`). Either pick that env's
kernel in VS Code (interpreter `/glade/work/manishrv/conda-envs/dev_gridgen/bin/python`), or
start Jupyter from it:

```bash
conda run -n dev_gridgen jupyter lab          # or: jupyter nbconvert --to notebook --execute ...
```

Run time is about 9 minutes, nearly all of it gridgen on the two annulus sectors and a deliberate
2 × 45 s demonstration of gridgen stalling. Figures go to `learn/figures/cmp_*.png`.

In [ ]:
import importlib.util
import os
import re
import subprocess
import sys
import tempfile
import textwrap
import time
import warnings
from pathlib import Path

import json
import numpy as np
import matplotlib
import matplotlib.ticker

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")  # running as a plain script
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
import shapely
from shapely.geometry import LinearRing

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # pygridgen's pkg_resources deprecation warning
    import pygridgen

WORKTREE = Path(
    "/glade/u/home/manishrv/documents/croc/regional_mom_workflows/mom6_forge-worktrees/conformal-solver"
)
LEARN = Path("/glade/u/home/manishrv/documents/croc/dev/mom6_forge_gridsketcher/learn")
FIG_DIR = LEARN / "figures"
FIG_DIR.mkdir(exist_ok=True)

spec = importlib.util.spec_from_file_location("cf", WORKTREE / "mom6_forge/_conformal.py")
cf = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cf)
print("Dan's solver from", cf.__file__)
print("pygridgen", pygridgen.__version__ if hasattr(pygridgen, "__version__") else "", pygridgen.__file__)

# The fixed plane for the synthetic cases (the one tests/test_conformal.py::get_rectangle uses).
PLANE = cf.MapProjection("lcc", -100.0, 35.0, 33.0, 37.0)
RASTERS = (250_000, 1_000_000)  # Dan's mesh sizes

## Helpers: running each solver

**Matching the two solvers' orientation.** Dan's `solve` reorders the 4 corners SW-first (the
corner with the smallest $x+y$), then counter-clockwise, and returns a supergrid indexed
`[j, i]` with `j` running S→N (side 0 → side 2) and `i` running W→E (side 3 → side 1).
gridgen's `ul_idx` vertex becomes the **upper-left** of its image rectangle, the side after it
(going CCW) the image's left side, and its lattice row 0 is the image's bottom. So passing
`ul_idx = cg.corners[3]` (Dan's NW corner) to gridgen makes gridgen's left = Dan's W, bottom =
Dan's S, and gridgen's `x[j, i]` lines up node for node with Dan's `x[j, i]` when
`shape = (2 ny + 1, 2 nx + 1)`. The code asserts this at the four corners.

**Modulus.** gridgen prints width/height of its image (`%.5g`); with the orientation above
that is $n_x/n_y$-like, i.e. **1 / Dan's `aspect`**. The full-precision value is recovered from
the image-corner coordinates gridgen prints at `verbose=2`.

**Timing.** gridgen is run twice on one object: a 3×3 lattice from cold (this is the CRDT
parameter solve, "solve"), then the full lattice warm-started from the cached σ ("map"). The C
library writes its log to file descriptor 2, which is captured.

In [ ]:
class CaptureStderr:
    """Capture what C code writes to file descriptor 2 (pygridgen's verbose log)."""

    def __enter__(self):
        sys.stderr.flush()
        self._tmp = tempfile.TemporaryFile(mode="w+b")
        self._saved = os.dup(2)
        os.dup2(self._tmp.fileno(), 2)
        return self

    def __exit__(self, *exc):
        sys.stderr.flush()
        os.dup2(self._saved, 2)
        os.close(self._saved)
        self._tmp.seek(0)
        self.text = self._tmp.read().decode(errors="replace")
        self._tmp.close()


def run_dan(x, y, corners, n_cells, raster_points, projection=PLANE, shape=None, mod=None):
    """Dan's solver on an outline given in plane metres (converted to lon/lat for its API).

    `shape=(nx, ny)` forces the cell counts, which `solve` otherwise rounds from its own aspect
    (so a finer mesh can flip ny by one); nothing else in the solve changes.
    """
    mod = mod or cf
    lon, lat = (np.asarray(a) for a in projection.to_lonlat(x, y))
    orig = mod._solve_mesh
    if shape is not None:
        def forced(*a, **k):
            f = orig(*a, **k)
            f["auto_nx_ny"] = (f["nx"], f["ny"])
            f["nx"], f["ny"] = shape
            return f
        mod._solve_mesh = forced
    try:
        t0 = time.perf_counter()
        cg = mod.solve(lon, lat, corners, n_cells, projection, raster_points=raster_points)
        return cg, time.perf_counter() - t0
    finally:
        mod._solve_mesh = orig


def run_gridgen(x, y, corners, ul_idx, shape, precision=1e-12):
    """gridgen on the same outline (metres; passed to gridgen in km). Returns a dict."""
    beta = np.zeros(len(x))
    beta[list(corners)] = 1.0  # +1 = a 90-degree image corner; 0 elsewhere (arc vertices too)
    out = dict(precision=precision)
    with CaptureStderr() as c1:
        t0 = time.perf_counter()
        g = pygridgen.Gridgen(
            np.asarray(x) / 1e3, np.asarray(y) / 1e3, beta, shape=(3, 3),
            ul_idx=int(ul_idx), verbose=2, precision=precision,
        )
        out["t_solve"] = time.perf_counter() - t0
    log = c1.text
    out["iter_errors"] = [float(e) for e in re.findall(r"^\s+\d+: error = (\S+)$", log, re.M)]
    last = log.split("image corners:\n")[-1]
    pts = re.findall(r"^\s+\d+: \(([-\d.e+]+), ([-\d.e+]+)\)$", last, re.M)[:4]
    pts = np.array(pts, dtype=float)
    out["modulus"] = np.ptp(pts[:, 0]) / np.ptp(pts[:, 1])  # image width / height
    out["modulus_printed"] = float(re.search(r"conformal modulus = (\S+)", log).group(1))
    out["n_vertices"] = int(re.findall(r"(\d+) vertices after phase 2 split", log)[-1])
    g.ny, g.nx = shape
    g.verbose = 1
    with CaptureStderr() as c2:
        t0 = time.perf_counter()
        g.generate_grid()
        out["t_map"] = time.perf_counter() - t0
    gen = c2.text.split("generating grid:")[-1]
    # one character per node: '.' secant inversion worked, 'o' fell back to the ODE, '-' failed
    out["n_ode"], out["n_fail"] = gen.count("o"), gen.count("-")
    gx = np.ma.filled(np.ma.asarray(g.x, dtype=float), np.nan) * 1e3
    gy = np.ma.filled(np.ma.asarray(g.y, dtype=float), np.nan) * 1e3
    out["x"], out["y"], out["n_nan"] = gx, gy, int(np.isnan(gx).sum())
    del g
    return out

## The five cases

All outlines are counter-clockwise polygons in plane metres. β = +1 at the 4 corners and 0 at
every other vertex. For the annulus sectors the exact map is $w=\log(z-c)$: radial lines at equal
angle, arcs at geometrically spaced radii. Each sector opens downward (centre above), so Dan's SW
corner is the outer-left one: **S = outer arc, E = right radial edge, N = inner arc, W = left
radial edge**, $\xi=(\theta-\theta_1)/\Delta\theta$, $\eta=\ln(r_2/r)/\ln(r_2/r_1)$, and the exact
`aspect` $=n_y/n_x=\ln(r_2/r_1)/\Delta\theta$ (gridgen's modulus is its reciprocal).

Arcs are polygonised at 0.5° per segment. The polygon's chords sit up to a sagitta
$r\,\delta^2/8$ inside the true arc (5–10 m here, under 0.002 of a cell), so "vs exact" has that
floor on boundary nodes; interior statistics (≥ 2 cells from the outline) don't feel it.

In [ ]:
def sector(r1, r2, dtheta_deg, seg_deg=0.5):
    """Annulus sector opening downward; returns x, y, corners, exact(xi, eta), exact aspect."""
    dth = np.radians(dtheta_deg)
    n_arc = int(round(dtheta_deg / seg_deg))
    t1 = 1.5 * np.pi - dth / 2
    yc = 0.5 * (r2 + r1 * np.cos(dth / 2))  # centre the shape on y = 0
    k = np.arange(n_arc + 1) / n_arc
    th_out, th_in = t1 + k * dth, t1 + dth - k * dth  # outer arc W->E, inner arc E->W
    x = np.r_[r2 * np.cos(th_out), r1 * np.cos(th_in)]
    y = yc + np.r_[r2 * np.sin(th_out), r1 * np.sin(th_in)]

    def exact(xi, eta):
        r, th = r2 * (r1 / r2) ** eta, t1 + xi * dth
        return r * np.cos(th), yc + r * np.sin(th)

    return x, y, [0, n_arc, n_arc + 1, 2 * n_arc + 1], exact, np.log(r2 / r1) / dth


def rectangle():
    x, y = np.array([-5e5, 5e5, 5e5, -5e5]), np.array([-3e5, -3e5, 3e5, 3e5])
    exact = lambda xi, eta: (-5e5 + 1e6 * xi, -3e5 + 6e5 * eta)
    return x, y, [0, 1, 2, 3], exact, 0.6


def trapezoid():  # as tests/test_conformal.py::test_straight_sides_have_smooth_edge_cells
    x, y = np.array([-5.5e5, 5.5e5, 5e5, -5e5]), np.array([-5e5, -5e5, 5e5, 5e5])
    return x, y, [0, 1, 2, 3], None, None


def california():
    d = json.loads((WORKTREE / "tests/data/california_bays.outline.json").read_text())["outline"]
    lon, lat = np.array(d["lon"]), np.array(d["lat"])
    proj = cf.projection_for("lcc", lon, lat)
    x, y = (np.asarray(a) for a in proj.to_xy(lon, lat))
    return x, y, d["corners"], None, None, proj


CASES = {}
for name, fn, n_cells, kw in [
    ("rectangle", rectangle, 6000, {}),
    ("trapezoid", trapezoid, 6400, {}),
    ("sector_2to1", lambda: sector(5e5, 1e6, 60), 6000, {}),
    ("sector_22to1", lambda: sector(5e5, 5.5e5, 120), 6000, {"precision": 1e-8}),
    ("california", california, 12000, {}),
]:
    r = fn()
    proj = r[5] if len(r) > 5 else PLANE
    x, y, corners, exact, aspect_exact = r[:5]
    assert cf._signed_area(x, y) > 0, "outlines are given counter-clockwise"
    CASES[name] = dict(x=x, y=y, corners=corners, exact=exact, aspect_exact=aspect_exact,
                       n_cells=n_cells, proj=proj, gg_kw=kw)
    print(f"{name:13s} {len(x):4d} vertices, n_cells {n_cells}, gridgen options {kw or 'defaults'}")

## Run both solvers

Dan's solver at 250k and 1M mesh points (`raster_points`); gridgen with pygridgen's defaults
(`nnodes=14`, `precision=1e-12`, Broyden on, thinning on), except **`precision=1e-8` on the 22:1
sector**, where 1e-12 never terminates (shown further down). gridgen's lattice is Dan's supergrid
shape at 250k, so every supergrid node is compared.

In [ ]:
RESULTS = {}
for name, c in CASES.items():
    res = RESULTS[name] = {}
    cg, res[f"t_dan_{RASTERS[0]}"] = run_dan(c["x"], c["y"], c["corners"], c["n_cells"], RASTERS[0], c["proj"])
    res[f"dan_{RASTERS[0]}"] = cg
    for rp in RASTERS[1:]:  # same nx, ny as at 250k, so nodes compare one to one
        res[f"dan_{rp}"], res[f"t_dan_{rp}"] = run_dan(
            c["x"], c["y"], c["corners"], c["n_cells"], rp, c["proj"], shape=(cg.nx, cg.ny))
        a = res[f"dan_{rp}"].aspect
        nx_auto = max(1, int(round(np.sqrt(c["n_cells"] / a))))
        if (nx_auto, int(round(nx_auto * a))) != (cg.nx, cg.ny):
            print(f"  note: at {rp} mesh points {name} would pick {nx_auto} x {int(round(nx_auto * a))} "
                  f"(aspect {a:.6f}); forced to {cg.nx} x {cg.ny}")
    shape = (2 * cg.ny + 1, 2 * cg.nx + 1)
    t0 = time.perf_counter()
    gg = res["gg"] = run_gridgen(c["x"], c["y"], c["corners"], cg.corners[3], shape, **c["gg_kw"])
    # Same corners at the same lattice corners (orientation check)
    P = np.column_stack([c["x"], c["y"]])
    for (j, i), k in zip([(0, 0), (0, -1), (-1, -1), (-1, 0)], cg.corners):
        assert np.hypot(gg["x"][j, i] - P[k, 0], gg["y"][j, i] - P[k, 1]) < 1.0, (name, j, i)
        assert np.hypot(cg.x[j, i] - P[k, 0], cg.y[j, i] - P[k, 1]) < 1e-3, (name, j, i)
    print(f"{name:13s} nx x ny = {cg.nx} x {cg.ny}; Dan {res[f't_dan_{RASTERS[0]}']:.1f} s / "
          f"{res[f't_dan_{RASTERS[1]}']:.1f} s; gridgen solve {gg['t_solve']:.1f} s + map "
          f"{gg['t_map']:.1f} s, {len(gg['iter_errors'])} iterations, {gg['n_vertices']} vertices")

## Dan's refined solver from #149, on California

PR #151 deliberately leaves out Dan's mesh refinement round singular vertices (`_spots`/`_levels`,
up to 5 halvings of the spacing near the sharpest vertices); it is planned as PR 2b. Dan's full
#149 solver (commit `55e7010b`) is read with `git show` into a temporary file and run on California
at 250k with the same 63 × 193 cells. Convex outlines never refine, so the other cases would be
identical to #151.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    src149 = subprocess.run(["git", "-C", str(WORKTREE), "show", "55e7010b:mom6_forge/_conformal.py"],
                            capture_output=True, text=True, check=True).stdout
    (Path(tmp) / "cf149.py").write_text(src149)
    spec149 = importlib.util.spec_from_file_location("cf149", Path(tmp) / "cf149.py")
    cf149 = importlib.util.module_from_spec(spec149)
    spec149.loader.exec_module(cf149)
c, res = CASES["california"], RESULTS["california"]
cg0 = res[f"dan_{RASTERS[0]}"]
proj149 = cf149.MapProjection(*(getattr(c["proj"], k) for k in ("kind", "lon_0", "lat_0", "lat_1", "lat_2")))
res["dan149"], res["t_dan149"] = run_dan(c["x"], c["y"], c["corners"], c["n_cells"], RASTERS[0], proj149,
                                         shape=(cg0.nx, cg0.ny), mod=cf149)
print(f"#149 refined, 250k: aspect {res['dan149'].aspect:.8f}, {res['t_dan149']:.1f} s")

## Measures

- **Local cell size** $h$ at each supergrid node: twice the geometric mean of the two supergrid
  spacings, from the reference grid (exact where known, else gridgen). Every distance below is in
  units of this $h$, i.e. "fractions of a cell".
- **Interior** = nodes at least 2 cells (4 supergrid nodes) from the outline.
- **Orthogonality** at interior q-nodes (cell corners): angle between the two grid lines through
  the node, from the supergrid half-cell neighbours (central differences). For the rectangle and
  the sectors the exact grid gives exactly 90° with this stencil; for curved grids the stencil
  itself has an $O(h^2)$ truncation, the same for both solvers. "Near corners" = the 3×3 interior
  q-nodes nearest each corner. "Boundary" = q-nodes on the sides (corners excluded), with a
  second-order one-sided difference inward (a plain one-sided step reads 0.16° even on the exact
  sector grid, which is the stencil, not the grid).
- **Boundary fit**: distance of every outer-row/column supergrid node from the outline polygon.

In [ ]:
def cell_size(X, Y):
    si = np.hypot(np.gradient(X, axis=1), np.gradient(Y, axis=1))
    sj = np.hypot(np.gradient(X, axis=0), np.gradient(Y, axis=0))
    return 2.0 * np.sqrt(si * sj)


def dist_stats(Xa, Ya, Xb, Yb, h):
    d = np.hypot(Xa - Xb, Ya - Yb) / h
    inner = d[4:-4, 4:-4]
    edge = np.r_[d[0], d[-1], d[1:-1, 0], d[1:-1, -1]]
    j, i = (int(v) for v in np.unravel_index(np.nanargmax(d), d.shape))
    return dict(d=d, max=np.nanmax(d), rms=np.sqrt(np.nanmean(d**2)), max_in=np.nanmax(inner),
                rms_in=np.sqrt(np.nanmean(inner**2)), max_edge=np.nanmax(edge), at=(j, i),
                max_km=np.nanmax(d * h) / 1e3, h_median_km=np.nanmedian(h) / 1e3)


def _angle_dev(ax, ay, bx, by):
    cos = (ax * bx + ay * by) / (np.hypot(ax, ay) * np.hypot(bx, by))
    return np.abs(90.0 - np.degrees(np.arccos(np.clip(cos, -1.0, 1.0))))


def ortho(X, Y):
    """Deviation from 90 deg: interior q-nodes (ny-1, nx-1), and boundary q-nodes."""
    r = c = slice(2, -2, 2)
    inner = _angle_dev(X[r, 3:-1:2] - X[r, 1:-3:2], Y[r, 3:-1:2] - Y[r, 1:-3:2],
                       X[3:-1:2, c] - X[1:-3:2, c], Y[3:-1:2, c] - Y[1:-3:2, c])
    # On the sides: the along-side central difference, and a second-order one-sided difference
    # inward (-3 z0 + 4 z1 - z2), so the stencil itself adds only O(h^2) to the angle.
    sides = []
    for a, b, k in ((0, 1, 2), (-1, -2, -3)):  # S and N sides
        sides.append(_angle_dev(X[a, 3:-1:2] - X[a, 1:-3:2], Y[a, 3:-1:2] - Y[a, 1:-3:2],
                                4 * X[b, c] - 3 * X[a, c] - X[k, c], 4 * Y[b, c] - 3 * Y[a, c] - Y[k, c]))
    for a, b, k in ((0, 1, 2), (-1, -2, -3)):  # W and E sides
        sides.append(_angle_dev(X[3:-1:2, a] - X[1:-3:2, a], Y[3:-1:2, a] - Y[1:-3:2, a],
                                4 * X[r, b] - 3 * X[r, a] - X[r, k], 4 * Y[r, b] - 3 * Y[r, a] - Y[r, k]))
    return inner, np.concatenate(sides)


def ortho_stats(X, Y):
    inner, edge = ortho(X, Y)
    near = [inner[:3, :3], inner[:3, -3:], inner[-3:, -3:], inner[-3:, :3]]  # SW SE NE NW
    return dict(p50=np.nanpercentile(inner, 50), p99=np.nanpercentile(inner, 99),
                max=np.nanmax(inner), corners=[float(np.nanmax(b)) for b in near],
                edge_p99=np.nanpercentile(edge, 99), edge_max=np.nanmax(edge), field=inner)


def boundary_fit(X, Y, x, y):
    ring = LinearRing(np.column_stack([x, y]))
    ox = np.r_[X[0], X[-1], X[:, 0], X[:, -1]]
    oy = np.r_[Y[0], Y[-1], Y[:, 0], Y[:, -1]]
    ok = np.isfinite(ox)
    return float(np.max(shapely.distance(shapely.points(ox[ok], oy[ok]), ring)))


M = {}
for name, c in CASES.items():
    res, m = RESULTS[name], {}
    cg0, gg = res[f"dan_{RASTERS[0]}"], res["gg"]
    if c["exact"] is not None:
        XI, ETA = np.meshgrid(np.linspace(0, 1, 2 * cg0.nx + 1), np.linspace(0, 1, 2 * cg0.ny + 1))
        EX, EY = c["exact"](XI, ETA)
        res["exact_xy"] = (EX, EY)
        h = cell_size(EX, EY)
    else:
        h = cell_size(gg["x"], gg["y"])
    res["h"] = h
    for rp in RASTERS:
        cg = res[f"dan_{rp}"]
        m[f"dan{rp}_vs_gg"] = dist_stats(cg.x, cg.y, gg["x"], gg["y"], h)
        if c["exact"] is not None:
            m[f"dan{rp}_vs_exact"] = dist_stats(cg.x, cg.y, EX, EY, h)
        m[f"ortho_dan{rp}"] = ortho_stats(cg.x, cg.y)
        m[f"fit_dan{rp}"] = boundary_fit(cg.x, cg.y, c["x"], c["y"])
    if c["exact"] is not None:
        m["gg_vs_exact"] = dist_stats(gg["x"], gg["y"], EX, EY, h)
        m["ortho_exact"] = ortho_stats(EX, EY)
    if "dan149" in res:
        cg = res["dan149"]
        m["dan149_vs_gg"] = dist_stats(cg.x, cg.y, gg["x"], gg["y"], h)
        m["ortho_dan149"] = ortho_stats(cg.x, cg.y)
        m["fit_dan149"] = boundary_fit(cg.x, cg.y, c["x"], c["y"])
    m["ortho_gg"] = ortho_stats(gg["x"], gg["y"])
    m["fit_gg"] = boundary_fit(gg["x"], gg["y"], c["x"], c["y"])
    M[name] = m
print("done")

## Tables

In [ ]:
R0, R1 = RASTERS
lab = {R0: "250k", R1: "1M"}


def fmt(v):
    return f"{v:.2e}" if (v != 0 and (abs(v) < 1e-2 or abs(v) >= 1e4)) else f"{v:.4g}"


lines = ["| case | nx × ny | exact aspect | Dan 250k | Dan 1M | gridgen 1/modulus | gridgen printed modulus | Dan 250k − gridgen (rel.) | Dan 1M − gridgen (rel.) |",
         "|---|---|---|---|---|---|---|---|---|"]
for name, c in CASES.items():
    res = RESULTS[name]
    a0, a1, ag = res[f"dan_{R0}"].aspect, res[f"dan_{R1}"].aspect, 1 / res["gg"]["modulus"]
    ex = f"{c['aspect_exact']:.8f}" if c["aspect_exact"] else "–"
    lines.append(f"| {name} | {res[f'dan_{R0}'].nx} × {res[f'dan_{R0}'].ny} | {ex} | {a0:.8f} | {a1:.8f} | "
                 f"{ag:.8f} | {res['gg']['modulus_printed']} | {fmt((a0 - ag) / ag)} | {fmt((a1 - ag) / ag)} |")
res = RESULTS["california"]
lines.append(f"| california, #149 refined 250k | | | {res['dan149'].aspect:.8f} | | {1 / res['gg']['modulus']:.8f} | | "
             f"{fmt((res['dan149'].aspect * res['gg']['modulus'] - 1))} | |")
print("Conformal modulus as Dan's aspect = ny/nx (gridgen prints the reciprocal)\n")
print("\n".join(lines))
for name, c in CASES.items():
    if c["aspect_exact"]:
        res = RESULTS[name]
        e = c["aspect_exact"]
        print(f"  {name}: rel. error vs exact: Dan 250k {fmt(res[f'dan_{R0}'].aspect / e - 1)}, "
              f"Dan 1M {fmt(res[f'dan_{R1}'].aspect / e - 1)}, gridgen {fmt(1 / res['gg']['modulus'] / e - 1)}")

In [ ]:
lines = ["| case | pair | max (all) | RMS (all) | max (interior) | RMS (interior) | max (outer ring) | where the max is (j, i) of (2ny, 2nx) | max (m) | median cell (km) |",
         "|---|---|---|---|---|---|---|---|---|---|"]
for name, m in M.items():
    cg = RESULTS[name][f"dan_{R0}"]
    pairs = [(f"dan{R0}_vs_gg", "Dan 250k − gridgen"), (f"dan{R1}_vs_gg", "Dan 1M − gridgen"),
             (f"dan{R0}_vs_exact", "Dan 250k − exact"), (f"dan{R1}_vs_exact", "Dan 1M − exact"),
             ("gg_vs_exact", "gridgen − exact"), ("dan149_vs_gg", "#149 refined 250k − gridgen")]
    for key, label in pairs:
        if key in m:
            s = m[key]
            lines.append(f"| {name} | {label} | {fmt(s['max'])} | {fmt(s['rms'])} | {fmt(s['max_in'])} | "
                         f"{fmt(s['rms_in'])} | {fmt(s['max_edge'])} | {s['at']} of ({2 * cg.ny}, {2 * cg.nx}) | "
                         f"{fmt(s['max_km'] * 1e3)} | {s['h_median_km']:.3g} |")
print("Node distance, in units of the local cell size\n")
print("\n".join(lines))

In [ ]:
lines = ["| case | grid | interior p50 (°) | p99 | max | max near SW / SE / NE / NW corners | boundary p99 | boundary max | boundary fit (m) |",
         "|---|---|---|---|---|---|---|---|---|"]
for name, m in M.items():
    rows = [("Dan 250k", m[f"ortho_dan{R0}"], m[f"fit_dan{R0}"]), ("Dan 1M", m[f"ortho_dan{R1}"], m[f"fit_dan{R1}"]),
            ("gridgen", m["ortho_gg"], m["fit_gg"])]
    if "ortho_exact" in m:
        rows.append(("exact", m["ortho_exact"], float("nan")))
    if "ortho_dan149" in m:
        rows.insert(2, ("#149 refined 250k", m["ortho_dan149"], m["fit_dan149"]))
    for label, o, fit in rows:
        cs = " / ".join(f"{v:.2g}" for v in o["corners"])
        lines.append(f"| {name} | {label} | {fmt(o['p50'])} | {fmt(o['p99'])} | {fmt(o['max'])} | {cs} | "
                     f"{fmt(o['edge_p99'])} | {fmt(o['edge_max'])} | {fmt(fit) if fit == fit else '–'} |")
print("Orthogonality: deviation from 90 degrees; boundary fit: max distance of outer nodes from the outline\n")
print("\n".join(lines))

In [ ]:
lines = ["| case | supergrid nodes | Dan 250k (s) | Dan 1M (s) | gridgen solve (s) | gridgen map (s) | gridgen vertices after splitting | iterations | final residual | precision | ODE-fallback nodes | NaN nodes |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|"]
for name, res in RESULTS.items():
    gg = res["gg"]
    lines.append(f"| {name} | {gg['x'].size} | {res[f't_dan_{R0}']:.1f} | {res[f't_dan_{R1}']:.1f} | {gg['t_solve']:.1f} | "
                 f"{gg['t_map']:.1f} | {gg['n_vertices']} | {len(gg['iter_errors'])} | {gg['iter_errors'][-1]:.2g} | "
                 f"{gg['precision']:g} | {gg['n_ode']} | {gg['n_nan']} |")
print(f"Wall time and gridgen diagnostics (#149 refined on California, 250k: {RESULTS['california']['t_dan149']:.1f} s)\n")
print("\n".join(lines))

### California: how far the corner disagreement reaches

Dan − gridgen with the nodes within $k$ cells (index distance) of any of the four corners left out,
and orthogonality of the interior cell corners within 3 cells of the most reflex non-corner vertex.

In [ ]:
c, res = CASES["california"], RESULTS["california"]
ang = cf.interior_angles(c["x"], c["y"])
non_corner = np.setdiff1d(np.arange(len(ang)), c["corners"])
reflex = int(non_corner[np.argmax(ang[non_corner])])
J, I = np.indices(res["gg"]["x"].shape)
corner_ji = [(0, 0), (0, J.shape[1] - 1), (J.shape[0] - 1, J.shape[1] - 1), (J.shape[0] - 1, 0)]
cdist = np.min([np.maximum(abs(J - a), abs(I - b)) for a, b in corner_ji], axis=0) / 2  # cells
print("| k (cells) | Dan 250k − gridgen max | RMS | Dan 1M − gridgen max | RMS | #149 refined 250k − gridgen max | RMS |")
print("|---|---|---|---|---|---|---|")
for k in (0, 2, 5, 10, 20):
    keep = cdist >= k
    row = [k]
    for key in [f"dan{rp}_vs_gg" for rp in RASTERS] + ["dan149_vs_gg"]:
        d = M["california"][key]["d"][keep]
        row += [fmt(np.nanmax(d)), fmt(np.sqrt(np.nanmean(d**2)))]
    print("| " + " | ".join(map(str, row)) + " |")
dd = np.hypot(res["gg"]["x"] - c["x"][reflex], res["gg"]["y"] - c["y"][reflex])
jr, ir = (int(v) for v in np.unravel_index(np.nanargmin(dd), dd.shape))
print(f"\nmost reflex non-corner vertex #{reflex}: {ang[reflex]:.0f} deg, nearest supergrid node (j, i) = ({jr}, {ir})")
qj, qi = jr // 2, ir // 2  # nearest q-node; interior q-node index = q-index - 1
for label, key in (("Dan 250k", f"ortho_dan{R0}"), ("Dan 1M", f"ortho_dan{R1}"), ("#149 refined", "ortho_dan149"),
                   ("gridgen", "ortho_gg")):
    f = M["california"][key]["field"]
    blk = f[max(qj - 4, 0):qj + 3, max(qi - 4, 0):qi + 3]
    print(f"  {label:9s} max |angle - 90| at interior cell corners within 3 cells of it: {np.nanmax(blk):.3g} deg")
print("corner angles (SW, SE, NE, NW):", [round(a) for a in res[f"dan_{R0}"].corner_angles])

## gridgen at its default precision on the 22:1 sector

gridgen's CRDT loop stops only when the residual norm reaches `precision`; it has **no iteration
cap** (gridgen.c, `find_sigmas`). On the 22:1 sector the residual stalls well above 1e-12, so at
the default precision the call either never returns or ends in gridgen's own error exit, which
calls `exit(1)` and would kill this kernel. Here each run is a subprocess killed after 45 s, and
the residual history is read from its verbose log. The sector is the same shape at coarser arc
spacing (4°, 2°, 1° per segment), to show where the stall starts.

In [ ]:
STALL_CODE = textwrap.dedent("""
    import sys, warnings, numpy as np
    warnings.simplefilter("ignore")
    import pygridgen
    x, y, corners, prec = np.load(sys.argv[1]), np.load(sys.argv[2]), [int(v) for v in sys.argv[3].split(",")], float(sys.argv[4])
    beta = np.zeros(len(x)); beta[corners] = 1.0
    pygridgen.Gridgen(x / 1e3, y / 1e3, beta, shape=(3, 3), ul_idx=corners[3], verbose=2, precision=prec)
""")
STALL = {}
with tempfile.TemporaryDirectory() as tmp:
    for seg in (4.0, 2.0, 1.0):
        x, y, corners, _, _ = sector(5e5, 5.5e5, 120, seg_deg=seg)
        np.save(f"{tmp}/x.npy", x), np.save(f"{tmp}/y.npy", y)
        logf = Path(tmp) / "log.txt"
        t0 = time.perf_counter()
        with open(logf, "w") as fh:
            try:
                proc = subprocess.run([sys.executable, "-I", "-c", STALL_CODE, f"{tmp}/x.npy", f"{tmp}/y.npy",
                                       ",".join(map(str, corners)), "1e-12"], stderr=fh, stdout=fh, timeout=45)
                outcome = "converged" if proc.returncode == 0 else "aborted"
            except subprocess.TimeoutExpired:
                outcome = "killed at 45 s"
        log = logf.read_text()
        errs = [float(e) for e in re.findall(r"^\s+\d+: error = (\S+)$", log, re.M)]
        msg = re.search(r"gridgen: error: (.*)", log)
        if outcome == "aborted" and msg:  # gridgen's quit() calls exit(1): it would take a kernel down
            outcome = f"aborted: {msg.group(1).strip()}"
        STALL[seg] = dict(errs=errs, outcome=outcome, n=len(x), t=time.perf_counter() - t0)
        tail = errs[len(errs) // 2:]
        print(f"{seg:.0f} deg segments ({len(x)} vertices): {outcome}, {len(errs)} iterations, "
              f"last {errs[-1]:.3g}, min over 2nd half {min(tail):.3g}, max over 2nd half {max(tail):.3g}")

## Figures

In [ ]:
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
SEQ = LinearSegmentedColormap.from_list(
    "seq_blue", ["#f3f7fd", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
SEQ.set_bad("#ffffff")
NORM = LogNorm(1e-6, 1e0)
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlesize": 9.5,
                     "figure.facecolor": "white"})


def q_lines(ax, X, Y, color, ls, lw, every=1, label=None):
    """Plot cell-corner (q) grid lines, every `every`-th line, with the outer ones always."""
    Xq, Yq = X[::2, ::2], Y[::2, ::2]
    ny, nx = Xq.shape
    rows = sorted(set(range(0, ny, every)) | {ny - 1})
    cols = sorted(set(range(0, nx, every)) | {nx - 1})
    for k, j in enumerate(rows):
        ax.plot(Xq[j] / 1e3, Yq[j] / 1e3, color=color, ls=ls, lw=lw, label=label if k == 0 else None)
    for i in cols:
        ax.plot(Xq[:, i] / 1e3, Yq[:, i] / 1e3, color=color, ls=ls, lw=lw)


def overlay(ax, name, title, every=1, window=None, lw=(0.9, 0.9)):
    res, c = RESULTS[name], CASES[name]
    gg, cg = res["gg"], res[f"dan_{R0}"]
    ax.fill(c["x"] / 1e3, c["y"] / 1e3, color="#ecebe8", zorder=0, lw=0)
    q_lines(ax, gg["x"], gg["y"], BLUE, "-", lw[0], every, "gridgen")
    q_lines(ax, cg.x, cg.y, ORANGE, (0, (3, 2)), lw[1], every, "Dan (250k mesh)")
    ax.plot(np.r_[c["x"], c["x"][:1]] / 1e3, np.r_[c["y"], c["y"][:1]] / 1e3, color=INK, lw=0.6)
    ax.set_aspect("equal")
    if window is not None:
        (x0, x1), (y0, y1) = window
        ax.set_xlim(x0 / 1e3, x1 / 1e3), ax.set_ylim(y0 / 1e3, y1 / 1e3)
    ax.set_title(title, loc="left")
    ax.set_xlabel("x (km)"), ax.set_ylabel("y (km)")


def dist_map(ax, name, key, title, index_space=False):
    res = RESULTS[name]
    d = M[name][key]["d"]
    X, Y = (res["exact_xy"] if "exact_xy" in res else (res["gg"]["x"], res["gg"]["y"]))
    if index_space:
        im = ax.imshow(np.clip(d, 1e-12, None), origin="lower", cmap=SEQ, norm=NORM, aspect="auto",
                       interpolation="nearest")
        ax.set_xlabel("i (supergrid, W→E)"), ax.set_ylabel("j (S→N)")
    else:
        im = ax.pcolormesh(X / 1e3, Y / 1e3, np.clip(d, 1e-12, None), cmap=SEQ, norm=NORM, shading="gouraud")
        ax.set_aspect("equal"), ax.set_xlabel("x (km)"), ax.set_ylabel("y (km)")
    s = M[name][key]
    ax.set_title(f"{title}\nmax {s['max']:.2g}, RMS {s['rms']:.2g}, interior max {s['max_in']:.2g}", loc="left")
    return im


def zoom_window(X, Y, j, i, cells=5):
    """A square window around supergrid node (j, i), `cells` local cells on each side."""
    win = np.s_[max(j - 2 * cells, 0):j + 2 * cells + 1, max(i - 2 * cells, 0):i + 2 * cells + 1]
    xs, ys = X[win], Y[win]
    cx, cy = X[j, i], Y[j, i]
    half = max(np.nanmax(np.abs(xs - cx)), np.nanmax(np.abs(ys - cy)))
    return (cx - half, cx + half), (cy - half, cy + half)


def finish(fig, im, name, axes):
    cb = fig.colorbar(im, ax=axes, shrink=0.85, extend="both", pad=0.02)
    cb.set_label("node distance / local cell size")
    save(fig, name)


def save(fig, name):
    fig.savefig(FIG_DIR / f"cmp_{name}.png", dpi=130, bbox_inches="tight")
    print("wrote", FIG_DIR / f"cmp_{name}.png")
    plt.show()  # inline in Jupyter; a no-op as a script
    plt.close(fig)

In [ ]:
# Rectangle and the moderate sector: overlay + three distance maps
for name, every in (("rectangle", 5), ("sector_2to1", 5)):
    fig, axs = plt.subplots(1, 4, figsize=(17, 3.9), layout="constrained")
    overlay(axs[0], name, f"{name}: every {every}th grid line", every=every)
    axs[0].legend(loc="lower left", fontsize=8, frameon=True)
    dist_map(axs[1], name, f"dan{R0}_vs_gg", "|Dan 250k − gridgen|")
    dist_map(axs[2], name, f"dan{R0}_vs_exact", "|Dan 250k − exact|")
    im = dist_map(axs[3], name, "gg_vs_exact", "|gridgen − exact|")
    finish(fig, im, name, axs[1:])

In [ ]:
# Trapezoid: overlay + Dan-gridgen at both mesh sizes
name = "trapezoid"
fig, axs = plt.subplots(1, 3, figsize=(13.5, 4.2), layout="constrained")
overlay(axs[0], name, "trapezoid: every 5th grid line", every=5)
axs[0].legend(loc="lower left", fontsize=8)
dist_map(axs[1], name, f"dan{R0}_vs_gg", "|Dan 250k − gridgen|")
im = dist_map(axs[2], name, f"dan{R1}_vs_gg", "|Dan 1M − gridgen|")
finish(fig, im, name, axs[1:])

In [ ]:
# The 22:1 sector: overlay, a zoom at its west end, distance maps in index space, convergence
name = "sector_22to1"
res = RESULTS[name]
fig = plt.figure(figsize=(16, 9.5), layout="constrained")
gs = fig.add_gridspec(3, 3, height_ratios=[1.15, 0.8, 0.8])
ax = fig.add_subplot(gs[0, :2])
gg, cg = res["gg"], res[f"dan_{R0}"]
c = CASES[name]
ax.fill(c["x"] / 1e3, c["y"] / 1e3, color="#ecebe8", zorder=0, lw=0)
for X, Y, col, ls, lab_ in ((gg["x"], gg["y"], BLUE, "-", "gridgen"), (cg.x, cg.y, ORANGE, (0, (3, 2)), "Dan (250k mesh)")):
    Xq, Yq = X[::2, ::2], Y[::2, ::2]
    for j in range(Xq.shape[0]):
        ax.plot(Xq[j] / 1e3, Yq[j] / 1e3, color=col, ls=ls, lw=0.5, label=lab_ if j == 0 else None)
    for i in list(range(0, Xq.shape[1], 10)) + [Xq.shape[1] - 1]:
        ax.plot(Xq[:, i] / 1e3, Yq[:, i] / 1e3, color=col, ls=ls, lw=0.5)
ax.set_aspect("equal"), ax.set_xlabel("x (km)"), ax.set_ylabel("y (km)")
ax.set_title("22:1 sector (r2/r1 = 1.1, 120°): every 10th line along the arc, every line across", loc="left")
ax.legend(loc="upper center", fontsize=8)
ax = fig.add_subplot(gs[0, 2])
overlay(ax, name, "zoom at the SW (outer-left) corner: every line",
        window=zoom_window(gg["x"], gg["y"], 0, 0, cells=12), lw=(1.0, 1.0))
axd = [fig.add_subplot(gs[1, :2]), fig.add_subplot(gs[2, :2])]
dist_map(axd[0], name, f"dan{R0}_vs_gg", "|Dan 250k − gridgen|", index_space=True)
im = dist_map(axd[1], name, "gg_vs_exact", "|gridgen − exact|", index_space=True)
ax = fig.add_subplot(gs[1:, 2])
for seg, col in zip(STALL, ("#1baf7a", ORANGE, "#6250d6")):
    s = STALL[seg]
    ax.semilogy(s["errs"], color=col, lw=1.2,
                label=f"{seg:.0f}° arcs ({s['n']} vertices), 1e-12: {s['outcome'].replace('aborted: ', 'aborted, ')}")
ax.semilogy(res["gg"]["iter_errors"], color=BLUE, lw=2,
            label=f"0.5° arcs ({len(c['x'])} vertices), precision 1e-8 (used)")
ax.axhline(1e-12, color=MUTED, lw=0.8, ls=":"), ax.axhline(1e-8, color=MUTED, lw=0.8, ls="--")
ax.text(1, 1.5e-12, "default precision 1e-12", color=MUTED, fontsize=8)
ax.text(1, 1.5e-8, "1e-8", color=MUTED, fontsize=8)
ax.set_xlabel("CRDT iteration (Broyden)"), ax.set_ylabel("residual ‖F‖₂")
ax.set_title("gridgen's residual on the 22:1 sector", loc="left")
ax.legend(fontsize=7.5, loc="upper right")
finish(fig, im, name, axd)

In [ ]:
# California: overlay, distance maps at both mesh sizes, and zooms at the two east corners and
# the most reflex non-corner vertex
name = "california"
res, c = RESULTS[name], CASES[name]
gg, cg = res["gg"], res[f"dan_{R0}"]
ang = cf.interior_angles(c["x"], c["y"])
non_corner = np.setdiff1d(np.arange(len(ang)), c["corners"])
reflex = int(non_corner[np.argmax(ang[non_corner])])
# the supergrid node nearest the reflex vertex (on the boundary)
dd = np.hypot(gg["x"] - c["x"][reflex], gg["y"] - c["y"][reflex])
jr, ir = np.unravel_index(np.nanargmin(dd), dd.shape)
print("corner angles (SW, SE, NE, NW):", [f"{a:.0f}" for a in cg.corner_angles],
      f"| most reflex non-corner vertex: #{reflex}, {ang[reflex]:.0f} deg, nearest node (j, i) = ({jr}, {ir})")
fig = plt.figure(figsize=(16, 9.8), layout="constrained")
gs = fig.add_gridspec(2, 4, height_ratios=[1.2, 1])
ax = fig.add_subplot(gs[0, 0])
overlay(ax, name, "California: every 6th line (#151, 250k)", every=6, lw=(0.7, 0.7))
ax.legend(loc="lower left", fontsize=8)
ax1 = fig.add_subplot(gs[0, 1])
dist_map(ax1, name, f"dan{R0}_vs_gg", "|Dan 250k − gridgen|")
ax2 = fig.add_subplot(gs[0, 2])
im = dist_map(ax2, name, f"dan{R1}_vs_gg", "|Dan 1M − gridgen|")
ax3 = fig.add_subplot(gs[0, 3])
im = dist_map(ax3, name, "dan149_vs_gg", "|#149 refined 250k − gridgen|")
fig.colorbar(im, ax=[ax1, ax2, ax3], shrink=0.6, extend="both", pad=0.02).set_label("node distance / local cell size")
zooms = [((0, -1), f"SE corner ({cg.corner_angles[1]:.0f}°)"), ((-1, -1), f"NE corner ({cg.corner_angles[2]:.0f}°)"),
         ((jr, ir), f"reflex vertex #{reflex} ({ang[reflex]:.0f}°)")]
for k, ((j, i), t) in enumerate(zooms):
    ax = fig.add_subplot(gs[1, k])
    jj, ii = j % gg["x"].shape[0], i % gg["x"].shape[1]
    overlay(ax, name, f"zoom: {t}, every line", window=zoom_window(gg["x"], gg["y"], jj, ii, cells=4), lw=(1.1, 1.1))
    q_lines(ax, res["dan149"].x, res["dan149"].y, "#1baf7a", (0, (1, 1.5)), 1.1, 1, "#149 refined (250k)")
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(4))
    if k == 0:
        ax.legend(loc="lower left", fontsize=7.5)
ax = fig.add_subplot(gs[1, 3])
d0, d1, d2 = M[name][f"dan{R0}_vs_gg"]["d"], M[name][f"dan{R1}_vs_gg"]["d"], M[name]["dan149_vs_gg"]["d"]
J, I = np.indices(d0.shape)
ring_dist = np.minimum.reduce([J, I, d0.shape[0] - 1 - J, d0.shape[1] - 1 - I]) / 2  # cells from the outline
for d, col, lab_ in ((d0, ORANGE, "#151 250k − gridgen"), (d1, "#6250d6", "#151 1M − gridgen"),
                     (d2, "#1baf7a", "#149 refined 250k − gridgen")):
    b = np.arange(0, np.floor(ring_dist.max()) + 1)
    sel = [d[(ring_dist >= v) & (ring_dist < v + 1)] for v in b]
    ax.semilogy(b, [np.nanmax(v) for v in sel], color=col, lw=1.5, label=lab_ + " (max)")
    ax.semilogy(b, [np.sqrt(np.nanmean(v**2)) for v in sel], color=col, lw=1, ls="--", label=lab_ + " (RMS)")
ax.set_xlabel("distance from the outline (cells, index space)"), ax.set_ylabel("node distance / cell size")
ax.set_title("California: Dan − gridgen by distance from the outline", loc="left")
ax.legend(fontsize=7.5)
save(fig, name)